# Option 4: Stock Market Anomaly Detection

## Objective
Detect unusual trading patterns and price anomalies

## Anomaly Detection Methods
- Isolation Forest
- Local Outlier Factor (LOF)
- One-Class SVM
- Z-Score based detection

## Detailed Plan
1. Load and filter target stocks for anomaly analysis.
2. Engineer anomaly-sensitive features (price shock, volume z-score, return z-score, momentum).
3. Normalize features for unsupervised models.
4. Train Isolation Forest, LOF, One-Class SVM, and statistical Z-score detector.
5. Build consensus anomaly voting to reduce false positives.
6. Visualize score distributions and stock time-series anomaly events.
7. Export anomaly records and stock-level anomaly rates.
8. Translate anomaly levels into practical trading risk actions.

In [ ]:
%pip install -q numpy pandas matplotlib seaborn scikit-learn kagglehub

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
import kagglehub, os
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✓ All libraries loaded successfully")

## Step 1: Data Loading & Exploration

## Step 2: Feature Engineering for Anomaly Detection

In [ ]:
# Download S&P 500 dataset
path = kagglehub.dataset_download("camnugent/sandp500")
files = os.listdir(path)
csv_file = [f for f in files if f.endswith('.csv')][0]
data = pd.read_csv(os.path.join(path, csv_file))

top_stocks = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA']
data_filtered = data[data['Name'].isin(top_stocks)].copy()
data_filtered = data_filtered.sort_values(['Name', 'date']).reset_index(drop=True)

print(f'Dataset shape: {data_filtered.shape}')
print(f'Stocks selected: {top_stocks}')
print(f'Date range: {data_filtered["date"].min()} to {data_filtered["date"].max()}')

In [ ]:
def create_anomaly_features(stock_data):
    stock_data = stock_data.sort_values('date').reset_index(drop=True)
    
    stock_data['price_change'] = stock_data['close'].pct_change()
    stock_data['intraday_range'] = (stock_data['high'] - stock_data['low']) / stock_data['open']
    stock_data['open_close_change'] = (stock_data['close'] - stock_data['open']) / stock_data['open']
    
    stock_data['volume_ma20'] = stock_data['volume'].rolling(20).mean()
    stock_data['volume_std20'] = stock_data['volume'].rolling(20).std()
    stock_data['volume_z'] = (stock_data['volume'] - stock_data['volume_ma20']) / stock_data['volume_std20']
    stock_data['volume_ratio'] = stock_data['volume'] / stock_data['volume_ma20']
    
    stock_data['return_ma20'] = stock_data['price_change'].rolling(20).mean()
    stock_data['return_std20'] = stock_data['price_change'].rolling(20).std()
    stock_data['return_z'] = (stock_data['price_change'] - stock_data['return_ma20']) / stock_data['return_std20']
    
    stock_data['momentum_3'] = stock_data['close'].pct_change(3)
    stock_data['momentum_5'] = stock_data['close'].pct_change(5)
    
    return stock_data

data_anomaly = pd.concat(
    [create_anomaly_features(data_filtered[data_filtered['Name'] == s]) for s in top_stocks],
    ignore_index=True
)
data_anomaly = data_anomaly.dropna().reset_index(drop=True)

feature_cols = [
    'price_change', 'intraday_range', 'open_close_change',
    'volume_z', 'volume_ratio', 'return_z',
    'momentum_3', 'momentum_5'
 ]

X = data_anomaly[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

DATA_DIR = '04_anomaly_detection/data'
GRAPH_DIR = '04_anomaly_detection/graph'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(GRAPH_DIR, exist_ok=True)

print(f'Feature matrix shape: {X_scaled.shape}')
print(f'Features used: {feature_cols}')

In [ ]:
# Train anomaly detection models
if_model = IsolationForest(contamination=0.05, random_state=42, n_estimators=200)
if_pred = if_model.fit_predict(X_scaled)
if_score = if_model.score_samples(X_scaled)

lof_model = LocalOutlierFactor(n_neighbors=20, contamination=0.05)
lof_pred = lof_model.fit_predict(X_scaled)
lof_score = lof_model.negative_outlier_factor_

svm_model = OneClassSVM(nu=0.05, kernel='rbf', gamma='scale')
svm_pred = svm_model.fit_predict(X_scaled)
svm_score = svm_model.decision_function(X_scaled).ravel()

z_anomaly = (np.abs(X_scaled).max(axis=1) > 3)

data_anomaly['if_anomaly'] = (if_pred == -1)
data_anomaly['lof_anomaly'] = (lof_pred == -1)
data_anomaly['svm_anomaly'] = (svm_pred == -1)
data_anomaly['z_anomaly'] = z_anomaly
data_anomaly['anomaly_votes'] = (
    data_anomaly['if_anomaly'].astype(int) +
    data_anomaly['lof_anomaly'].astype(int) +
    data_anomaly['svm_anomaly'].astype(int) +
    data_anomaly['z_anomaly'].astype(int)
)

print('Anomalies detected by method:')
print(f"Isolation Forest: {(if_pred == -1).sum()}")
print(f"LOF: {(lof_pred == -1).sum()}")
print(f"One-Class SVM: {(svm_pred == -1).sum()}")
print(f"Z-Score: {z_anomaly.sum()}")
print(f"Consensus (>=3 votes): {(data_anomaly['anomaly_votes'] >= 3).sum()}")

## Step 3: Anomaly Score Distribution and Method Comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0,0].hist(if_score, bins=60, color='steelblue', alpha=0.75, edgecolor='black')
axes[0,0].set_title('Isolation Forest Score Distribution')
axes[0,0].set_xlabel('Score (lower = more anomalous)')
axes[0,0].grid(alpha=0.3)

axes[0,1].hist(lof_score, bins=60, color='coral', alpha=0.75, edgecolor='black')
axes[0,1].set_title('LOF Score Distribution')
axes[0,1].set_xlabel('Score (lower = more anomalous)')
axes[0,1].grid(alpha=0.3)

axes[1,0].hist(svm_score, bins=60, color='seagreen', alpha=0.75, edgecolor='black')
axes[1,0].set_title('One-Class SVM Decision Score')
axes[1,0].set_xlabel('Score (lower = more anomalous)')
axes[1,0].grid(alpha=0.3)

vote_counts = data_anomaly['anomaly_votes'].value_counts().sort_index()
axes[1,1].bar(vote_counts.index, vote_counts.values, color='mediumpurple', edgecolor='black')
axes[1,1].set_title('Consensus Vote Distribution')
axes[1,1].set_xlabel('Number of Models Flagging Anomaly')
axes[1,1].set_ylabel('Count')
axes[1,1].set_xticks([0,1,2,3,4])
axes[1,1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f'{GRAPH_DIR}/01_anomaly_score_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

method_counts = pd.Series({
    'Isolation Forest': (if_pred == -1).sum(),
    'LOF': (lof_pred == -1).sum(),
    'One-Class SVM': (svm_pred == -1).sum(),
    'Z-Score': z_anomaly.sum(),
    'Consensus (>=3)': (data_anomaly['anomaly_votes'] >= 3).sum()
})

fig, ax = plt.subplots(figsize=(10, 5))
method_counts.sort_values().plot(kind='barh', color='slateblue', edgecolor='black', ax=ax)
ax.set_title('Anomaly Detection Count by Method', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of anomalies')
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(f'{GRAPH_DIR}/02_method_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## Step 4: Time-Series View of Anomalies (Example: AAPL)

In [ ]:
aapl = data_anomaly[data_anomaly['Name'] == 'AAPL'].copy().reset_index(drop=True)
aapl_if = aapl['if_anomaly'].values
aapl_consensus = (aapl['anomaly_votes'] >= 3).values

fig, axes = plt.subplots(3, 1, figsize=(16, 11), sharex=True)

axes[0].plot(aapl['close'].values, color='steelblue', linewidth=1.2, label='Close Price')
axes[0].scatter(np.where(aapl_if)[0], aapl.loc[aapl_if, 'close'].values, c='red', s=45, label='IF anomaly')
axes[0].set_title('AAPL Close Price with Anomalies', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Price ($)')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].bar(np.arange(len(aapl)), aapl['volume'].values, color='gray', alpha=0.6, label='Volume')
axes[1].scatter(np.where(aapl_consensus)[0], aapl.loc[aapl_consensus, 'volume'].values, c='darkorange', s=40, label='Consensus anomaly')
axes[1].set_title('AAPL Volume with Consensus Anomalies', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Volume')
axes[1].legend()
axes[1].grid(alpha=0.3, axis='y')

axes[2].plot(aapl['price_change'].values * 100, color='seagreen', linewidth=1.2, label='Daily Return (%)')
axes[2].axhline(0, color='black', linewidth=0.8)
axes[2].scatter(np.where(aapl_consensus)[0], aapl.loc[aapl_consensus, 'price_change'].values * 100, c='purple', s=40, label='Consensus anomaly')
axes[2].set_title('AAPL Daily Return with Consensus Anomalies', fontsize=12, fontweight='bold')
axes[2].set_ylabel('Return (%)')
axes[2].set_xlabel('Time index')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{GRAPH_DIR}/03_aapl_timeseries_anomalies.png', dpi=300, bbox_inches='tight')
plt.show()

## Step 5: Save Outputs and Trading Risk Plan

In [ ]:
# Save detailed anomaly records
data_anomaly['consensus_anomaly'] = data_anomaly['anomaly_votes'] >= 3
anomaly_records = data_anomaly[data_anomaly['consensus_anomaly']].copy()
anomaly_records = anomaly_records.sort_values(['Name', 'date'])

anomaly_records.to_csv(f'{DATA_DIR}/consensus_anomalies.csv', index=False)
data_anomaly.to_csv(f'{DATA_DIR}/all_scored_records.csv', index=False)

# Stock-level anomaly summary
stock_summary = data_anomaly.groupby('Name').agg(
    total_days=('Name', 'count'),
    if_anomalies=('if_anomaly', 'sum'),
    lof_anomalies=('lof_anomaly', 'sum'),
    svm_anomalies=('svm_anomaly', 'sum'),
    z_anomalies=('z_anomaly', 'sum'),
    consensus_anomalies=('consensus_anomaly', 'sum')
).reset_index()
stock_summary['consensus_rate_%'] = (stock_summary['consensus_anomalies'] / stock_summary['total_days'] * 100).round(2)
stock_summary.to_csv(f'{DATA_DIR}/stock_anomaly_summary.csv', index=False)

fig, ax = plt.subplots(figsize=(10, 5))
plot_df = stock_summary.sort_values('consensus_rate_%', ascending=False)
ax.bar(plot_df['Name'], plot_df['consensus_rate_%'], color='teal', edgecolor='black')
ax.set_title('Consensus Anomaly Rate by Stock', fontsize=13, fontweight='bold')
ax.set_xlabel('Stock')
ax.set_ylabel('Consensus Anomaly Rate (%)')
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(f'{GRAPH_DIR}/04_stock_consensus_rate.png', dpi=300, bbox_inches='tight')
plt.show()

print('Saved data files:')
print('- consensus_anomalies.csv')
print('- all_scored_records.csv')
print('- stock_anomaly_summary.csv')

print('\nTrading Risk Plan (for anomaly events):')
print('1. Green (0-1 votes): normal position sizing')
print('2. Yellow (2 votes): reduce position by 30%')
print('3. Orange (3 votes): reduce position by 60%, tighten stop-loss')
print('4. Red (4 votes): close position and review event/news immediately')